# 07 — Figures & Tables

Generate all final figures for the blog post and export to `blog/assets/`.

In [ ]:
import sys
sys.path.insert(0, '..')

import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

from src.utils import RESULTS_DIR, BLOG_ASSETS
from src.plotting import COLORS

os.makedirs(BLOG_ASSETS, exist_ok=True)

matplotlib.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300,
    'font.size': 12, 'axes.titlesize': 14, 'axes.labelsize': 12,
    'legend.fontsize': 11, 'figure.figsize': (8, 5),
})
%matplotlib inline

In [ ]:
# Load all results
with open(os.path.join(RESULTS_DIR, 'curves_data.json')) as f:
    curves = json.load(f)
with open(os.path.join(RESULTS_DIR, 'stability_results.json')) as f:
    stability = json.load(f)
with open(os.path.join(RESULTS_DIR, 'ablation_results.json')) as f:
    ablation = json.load(f)
with open(os.path.join(RESULTS_DIR, 'coldstart_results.json')) as f:
    coldstart = json.load(f)
with open(os.path.join(RESULTS_DIR, 'gcn_base_s42_metrics.json')) as f:
    gcn_m = json.load(f)
with open(os.path.join(RESULTS_DIR, 'rgcn_base_s42_metrics.json')) as f:
    rgcn_m = json.load(f)
with open(os.path.join(RESULTS_DIR, 'heuristic_base_metrics.json')) as f:
    heur_m = json.load(f)

print('All results loaded.')

## Figure 1: ROC & PR Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for name, c in curves.items():
    color = COLORS.get(name, '#999')
    fpr, tpr = np.array(c['fpr']), np.array(c['tpr'])
    label = name.upper()
    if name == 'gcn': label += f" (AUROC={gcn_m['auroc']:.3f})"
    elif name == 'rgcn': label += f" (AUROC={rgcn_m['auroc']:.3f})"
    else: label += f" (AUROC={heur_m['auroc']:.3f})"
    ax1.plot(fpr, tpr, label=label, color=color, linewidth=2)
ax1.plot([0,1], [0,1], 'k--', alpha=0.3)
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curves')
ax1.legend(loc='lower right')

for name in ['gcn', 'rgcn']:
    c = curves[name]
    color = COLORS.get(name)
    rec, pre = np.array(c['recall']), np.array(c['precision'])
    auprc = gcn_m['auprc'] if name == 'gcn' else rgcn_m['auprc']
    ax2.plot(rec, pre, label=f"{name.upper()} (AUPRC={auprc:.3f})", color=color, linewidth=2)
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curves')
ax2.legend()

plt.tight_layout()
fig.savefig(os.path.join(BLOG_ASSETS, 'fig1_accuracy.png'), dpi=300, bbox_inches='tight')
plt.show()

## Figure 2: Stability Score Curves (Gaussian Noise)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for model_name in ['gcn', 'rgcn']:
    data = stability['gaussian'][model_name]
    sigmas = sorted(data.keys(), key=float)
    means = [data[s]['stability_score_mean'] for s in sigmas]
    stds = [data[s]['stability_score_std'] for s in sigmas]
    x = [float(s) for s in sigmas]
    means, stds = np.array(means), np.array(stds)
    ax.plot(x, means, 'o-', label=model_name.upper(), color=COLORS[model_name], linewidth=2)
    ax.fill_between(x, means - stds, means + stds, alpha=0.15, color=COLORS[model_name])

ax.set_xlabel('Perturbation Strength (σ relative to embedding norm)')
ax.set_ylabel('Stability Score')
ax.set_title('Embedding Stability Under Gaussian Noise')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=12)
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(BLOG_ASSETS, 'fig2_stability.png'), dpi=300, bbox_inches='tight')
plt.show()

## Figure 3: Top-K Overlap Degradation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, k in zip(axes, [10, 20, 50]):
    for model_name in ['gcn', 'rgcn']:
        data = stability['gaussian'][model_name]
        sigmas = sorted(data.keys(), key=float)
        x = [float(s) for s in sigmas]
        means = [data[s][f'mean_jaccard_top{k}_mean'] for s in sigmas]
        ax.plot(x, means, 'o-', label=model_name.upper(), color=COLORS[model_name], linewidth=2)
    ax.set_xlabel('σ (relative)')
    ax.set_ylabel(f'Mean Jaccard (Top-{k})')
    ax.set_title(f'Top-{k} Overlap')
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(BLOG_ASSETS, 'fig3_topk.png'), dpi=300, bbox_inches='tight')
plt.show()

## Figure 4: Edge-Type Ablation

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

variants = ['full_rgcn', 'no_drug_gene', 'no_disease_gene', 'no_disease_drug']
labels = ['Full RGCN', '- Drug-Gene', '- Disease-Gene', '- Disease-Drug']
x = np.arange(len(variants))
width = 0.35

aurocs = [ablation[v]['auroc'] for v in variants]
stab_scores = [ablation[v]['stability_score_01'] for v in variants]

bars1 = ax.bar(x - width/2, aurocs, width, label='AUROC', color='#4c72b0')
bars2 = ax.bar(x + width/2, stab_scores, width, label='Stability Score (σ=0.10)', color='#dd8452')

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel('Score')
ax.set_title('Edge-Type Removal Ablation (RGCN)')
ax.legend(fontsize=11)
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.3)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
fig.savefig(os.path.join(BLOG_ASSETS, 'fig4_ablation.png'), dpi=300, bbox_inches='tight')
plt.show()

## Figure 5: Cold-Start Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

metrics_to_show = ['auroc', 'auprc', 'stability_score_01']
metric_labels = ['AUROC', 'AUPRC', 'Stability Score']
x = np.arange(len(metrics_to_show))
width = 0.2

gcn_std = [gcn_m.get(m, 0) for m in metrics_to_show]
gcn_std[2] = stability['gaussian']['gcn']['0.1']['stability_score_mean']
rgcn_std = [rgcn_m.get(m, 0) for m in metrics_to_show]
rgcn_std[2] = stability['gaussian']['rgcn']['0.1']['stability_score_mean']
gcn_cold_vals = [coldstart['gcn_cold'].get(m, 0) for m in metrics_to_show]
rgcn_cold_vals = [coldstart['rgcn_cold'].get(m, 0) for m in metrics_to_show]

ax.bar(x - 1.5*width, gcn_std, width, label='GCN (Standard)', color=COLORS['gcn'], alpha=0.9)
ax.bar(x - 0.5*width, gcn_cold_vals, width, label='GCN (Cold-Start)', color=COLORS['gcn'], alpha=0.4, hatch='//')
ax.bar(x + 0.5*width, rgcn_std, width, label='RGCN (Standard)', color=COLORS['rgcn'], alpha=0.9)
ax.bar(x + 1.5*width, rgcn_cold_vals, width, label='RGCN (Cold-Start)', color=COLORS['rgcn'], alpha=0.4, hatch='//')

ax.set_xticks(x)
ax.set_xticklabels(metric_labels, fontsize=12)
ax.set_ylabel('Score')
ax.set_title('Standard vs Cold-Start Performance')
ax.legend(fontsize=10, loc='upper right')
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(BLOG_ASSETS, 'fig5_coldstart.png'), dpi=300, bbox_inches='tight')
plt.show()

## Figure 6: Summary Results Table

In [ ]:
gauss_gcn = stability['gaussian']['gcn']
gauss_rgcn = stability['gaussian']['rgcn']

summary_df = pd.DataFrame({
    'Model': ['Heuristic', 'GCN', 'RGCN'],
    'AUROC': [heur_m['auroc'], gcn_m['auroc'], rgcn_m['auroc']],
    'AUPRC': [heur_m['auprc'], gcn_m['auprc'], rgcn_m['auprc']],
    'Hits@20': ['-', gcn_m.get('hits@20', '-'), rgcn_m.get('hits@20', '-')],
    'SS (σ=0.05)': ['-', f"{gauss_gcn['0.05']['stability_score_mean']:.3f}",
                         f"{gauss_rgcn['0.05']['stability_score_mean']:.3f}"],
    'SS (σ=0.10)': ['-', f"{gauss_gcn['0.1']['stability_score_mean']:.3f}",
                         f"{gauss_rgcn['0.1']['stability_score_mean']:.3f}"],
    'SS (σ=0.20)': ['-', f"{gauss_gcn['0.2']['stability_score_mean']:.3f}",
                         f"{gauss_rgcn['0.2']['stability_score_mean']:.3f}"],
})

for col in ['AUROC', 'AUPRC']:
    summary_df[col] = summary_df[col].apply(lambda x: f"{x:.3f}" if isinstance(x, float) else x)
for col in ['Hits@20']:
    summary_df[col] = summary_df[col].apply(lambda x: f"{x:.3f}" if isinstance(x, float) else x)

display(summary_df)

# Save as HTML for blog
html = summary_df.to_html(index=False, classes='results-table', border=0)
with open(os.path.join(BLOG_ASSETS, 'fig6_table.html'), 'w') as f:
    f.write(html)
print('Table saved.')